# MapAction map automation

Builds MapAction map products automatically. For each map it makes an ArcGIS Pro project from the map template and a layout, adds and styles the layers, zooms the maps, fills in the legend and marginalia text, and exports a JPEG.

**How to run:** run the cells one at a time from the top (click a cell, then **Ctrl+Enter**), or choose **Run > Run All Cells**. Some cells ask questions: type the answer in the box that appears and press **Enter**.

**Before you start:**
- **Open `map_automation_pro.aprx` first.** It is an empty ArcGIS Pro project in the same folder as this notebook (`GIS/7_Python/71_Jupyter/map_automation`, next to `map_automation_pro_lib.py` and `map_automation_pro_config.py`). Open it in ArcGIS Pro, then open this notebook from its Catalog pane (**Folders > map_automation**). ArcGIS Pro runs a notebook from the open project's folder, so this is how the notebook finds its files.
- Settings such as which maps to build (`MAPS_TO_RUN`) and testing mode (`TEST_MODE`) are in `map_automation_pro_config.py`. Open it in a text editor to change them, then run this notebook again from the top.
- Event details (GLIDE number, language, disclaimer...) come from the project's `event_description.json`.

Outputs go to `GIS/3_Mapping/33_Map_Projects/<map code>/` (projects) and `GIS/3_Mapping/34_Map_Products_MapAction/<map code>/<version>/` (JPEGs).

## 1. Load the tools and find the project

Loads the map automation functions and settings, finds the project folder this notebook is in, reads its event details and checks the settings.

`PROJECT_FOLDER` can usually stay as `None`. To build maps for a different project, set it to that project's folder, e.g. `r"C:\Projects\2026\2026-ven-001"`.

In [ ]:
PROJECT_FOLDER = None

# Reload, so changes to the settings file are used without restarting the notebook.
import importlib
try:
    import map_automation_pro_config
    import map_automation_pro_lib as automation
except ModuleNotFoundError:
    raise SystemExit("Map automation files not found. Open map_automation_pro.aprx (in this notebook's folder) "
                     "in ArcGIS Pro first, then open this notebook from its Catalog pane.") from None
importlib.reload(map_automation_pro_config)
importlib.reload(automation)

run = automation.start_run(PROJECT_FOLDER)

## 2. Find the data

Finds the dataset in `GIS/2_Active_Data` for each layer of the maps. If several datasets match a layer, the settings' preferences are used; if it is still unclear, you are asked to choose.

It also makes the *feather*: the layer that fades everything outside the affected country.

In [ ]:
data = automation.find_data(run)

## 3. Choose projections, layout and label language

Four questions, used for every map of this run (any answered in the settings file are skipped):

1. **Main map projection:** ArcGIS Pro default (latitude/longitude), a UTM zone (e.g. `19N`; see the Data Scramble Sheet), or an EPSG/ESRI code.
2. **Location map projection:** the same options, or `0` for the Main map's.
3. **Layout:** for initial maps, pick one matching the country's shape; the question says how wide the country is compared with its height.
4. **Admin label language:** the `xx` in the `ADM1_xx` / `ADM2_xx` name fields used for admin 1 and 2 labels (e.g. `EN`, `ES`).

In [ ]:
choices = automation.ask_questions(run, data)

## 4. Check for problems

Lists anything missing, such as layer files, data or label fields. The maps are still built, but check any warnings before using them.

In [ ]:
automation.check_before_building(run, data, choices)

## 5. Clear old outputs (testing mode only)

If `TEST_MODE` is on in the settings, this deletes all previous outputs of this notebook, so the new maps are version `v01`. Testing mode only works in a test copy of a project (one with a `TEST_ENVIRONMENT.txt` file). If it is off, nothing is deleted.

Close any of the old map projects in ArcGIS Pro first.

In [ ]:
automation.clear_old_outputs(run)

## 6. Build the maps

Builds, exports and tidies each map. This takes around half a minute per map. When it finishes, the project (`.aprx`) and JPEG of each map are listed.

In [ ]:
automation.build_maps(run, data, choices)